<a href="https://colab.research.google.com/github/SALMA-elbardi/churn_sentinel/blob/main/laby.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# JEU DU LABYRINTHE - Apprentissage par Renforcement
# ============================================

#===================================================#
#        Vous êtes les bienvenus !           =======#
#         Allons jouer ensemble !🎮         =======#
#===================================================#

import numpy as np
from IPython.display import display, clear_output
import ipywidgets as widgets

# --- Grille secrète ---
# 0 = libre, 1 = mur, 2 = sortie
grille = np.array([
    [0, 0, 1, 0, 0],  # ligne 1
    [0, 1, 0, 0, 1],  # ligne 2
    [0, 0, 1, 0, 0],  # ligne 3
    [1, 0, 0, 0, 2],  # ligne 4
])

LIGNES, COLS = grille.shape
DEPART = (0, 0)
SORTIE = (3, 4)

# --- État du jeu ---
position = list(DEPART)
historique = []
score_total = 0
pas = 0

# --- Affichage ---
output = widgets.Output()
tableau_output = widgets.Output()

def afficher_grille():
    with output:
        clear_output(wait=True)
        print("=" * 35)
        print(f"  Position : ({position[0]+1}, {position[1]+1})")
        print(f"  Score total : {score_total}")
        print(f"  Pas : {pas}")
        print("=" * 35)
        print()
        print("     ", end="")
        for c in range(COLS):
            print(f"  {c+1} ", end="")
        print()
        for r in range(LIGNES):
            print(f"  {r+1}  ", end="")
            for c in range(COLS):
                if [r, c] == position:
                    print(" 🤖 ", end="")
                elif (r, c) == SORTIE and grille[r][c] == 2 and any(
                    h['position_apres'] == list(SORTIE) for h in historique
                ):
                    print(" 🏁 ", end="")
                else:
                    print(" ·  ", end="")
            print()
        print()

def afficher_tableau():
    with tableau_output:
        clear_output(wait=True)
        print(f"{'Pas':>3} | {'Position':>10} | {'Action':>8} | "
              f"{'Résultat':>12} | {'Points':>6} | {'Total':>6}")
        print("-" * 65)
        for h in historique:
            print(f"{h['pas']:>3} | "
                  f"({h['pos_avant'][0]+1},{h['pos_avant'][1]+1}):>10 | "
                  f"{h['action']:>8} | "
                  f"{h['resultat']:>12} | "
                  f"{h['points']:>+6} | "
                  f"{h['total']:>6}")

def jouer(action_nom, dr, dc):
    global position, score_total, pas

    pos_avant = position.copy()
    nr, nc = position[0] + dr, position[1] + dc

    if nr < 0 or nr >= LIGNES or nc < 0 or nc >= COLS:
        points = -3
        resultat = "HORS GRILLE"
    # Mur
    elif grille[nr][nc] == 1:
        points = -3
        resultat = "MUR !"
    # Sortie
    elif grille[nr][nc] == 2:
        position[0], position[1] = nr, nc
        points = 10
        resultat = "SORTIE !"
    # Case libre
    else:
        position[0], position[1] = nr, nc
        points = -1
        resultat = "OK"

    pas += 1
    score_total += points

    historique.append({
        'pas': pas,
        'pos_avant': pos_avant,
        'action': action_nom,
        'position_apres': position.copy(),
        'resultat': resultat,
        'points': points,
        'total': score_total
    })

    afficher_grille()
    afficher_tableau()

    if resultat == "SORTIE !":
        with output:
            print("🎉 BRAVO ! Sortie trouvée !")
            print(f"Score final : {score_total} en {pas} pas")

def recommencer(_):
    """Remet à zéro"""
    global position, score_total, pas, historique
    position = list(DEPART)
    score_total = 0
    pas = 0
    historique = []
    afficher_grille()
    afficher_tableau()

# --- Boutons ---
btn_haut = widgets.Button(description="↑ Haut", layout=widgets.Layout(width='80px'))
btn_bas = widgets.Button(description="↓ Bas", layout=widgets.Layout(width='80px'))
btn_gauche = widgets.Button(description="← Gauche", layout=widgets.Layout(width='80px'))
btn_droite = widgets.Button(description="→ Droite", layout=widgets.Layout(width='80px'))
btn_reset = widgets.Button(description="🔄 Recommencer",
                           button_style='warning',
                           layout=widgets.Layout(width='120px'))

btn_haut.on_click(lambda _: jouer("haut", -1, 0))
btn_bas.on_click(lambda _: jouer("bas", 1, 0))
btn_gauche.on_click(lambda _: jouer("gauche", 0, -1))
btn_droite.on_click(lambda _: jouer("droite", 0, 1))
btn_reset.on_click(recommencer)

# --- Interface ---
boutons = widgets.VBox([
    widgets.HBox([widgets.Label(''), btn_haut], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([btn_gauche, widgets.Label('  '), btn_droite], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([widgets.Label(''), btn_bas], layout=widgets.Layout(justify_content='center')),
    widgets.HBox([btn_reset], layout=widgets.Layout(justify_content='center'))
])

titre = widgets.HTML("<h2>🎮 Labyrinthe RL - Motivation</h2>"
                     "<p>Vous êtes le robot 🤖. Trouvez la sortie !</p>"
                     "<p><b>Règles :</b> déplacement libre = -1, "
                     "mur/bord = -3 (reste sur place), sortie = +10</p>")

display(titre)
display(widgets.HBox([output, boutons]))
display(tableau_output)

# Lancement
afficher_grille()

HTML(value='<h2>🎮 Labyrinthe RL - Motivation</h2><p>Vous êtes le robot 🤖. Trouvez la sortie !</p><p><b>Règles …

Output()